# EN4554 Assignment 1, Part 1: LoRA+ on SmolVLM-256M
## Notebook 01: setup, data subsets and zero-shot baseline

**This notebook:**
1. Build fixed, reproducible data subsets and save them to Google Drive:
   * **TextVQA** (the *hard* task): 2,000 train questions from the official `train` split, plus 500 eval questions from the official `validation` split.
   * **VQAv2** (the *easy* task): 2,000 train questions from the official `train` split, plus 500 eval questions from the official `validation` split.
2. Load **SmolVLM-256M-Instruct** and measure its **zero-shot** VQA accuracy on both eval sets.
   * This is our "no fine-tuning" baseline.
   * It also checks the assumption behind H2: is TextVQA really harder for this model than VQAv2?

Runtime used: **GPU (T4)**.

In [ ]:
!pip install -q "transformers==5.19.0" "peft==0.21.2" "datasets==5.1.0" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 14.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
import os, sys, re, json, time, random
import numpy as np
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/EN4554_LoRAplus"
else:
    ROOT = "./EN4554_LoRAplus"
DATA_DIR = f"{ROOT}/data"
RES_DIR = f"{ROOT}/results"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)

SEED = 42
N_TRAIN = 2000       # training questions per task
N_EVAL = 500         # evaluation questions per task
IMG_MAX = 512        # SmolVLM sees a single 512x512 image when tiling is off, so store images at this size
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

import transformers, peft, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| datasets", datasets.__version__, "| device", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

Mounted at /content/drive
torch 2.11.0+cu130 | transformers 5.19.0 | peft 0.21.2 | datasets 5.1.0 | device cuda
Tesla T4


## 1. Build the data subsets
Each dataset is stored on the Hugging Face Hub as many data files (parquet files, about 100–300 MB each). We:

1. pick a few files at random (with `SEED`, so the choice is **reproducible**) and download only those, to Colab's local disk, not RAM;
2. read their question/answer columns, and randomly choose `N_TRAIN` or `N_EVAL` questions;
3. read the images of the chosen questions **one file section at a time**, shrink them to 512 px (the resolution the model sees anyway), and write them straight to disk.

Memory stays at a few hundred MB. *(An earlier version used streaming with a shuffle buffer. That used about 3× the downloaded size in RAM and crashed the free Colab runtime.)*

Both tasks use the **official splits**: train questions come from `train`, eval questions from `validation`. So no image is shared between training and evaluation.
The subsets are saved to Drive, so every later notebook (and every LoRA / LoRA+ run) uses **exactly the same questions**. If they already exist, this cell just loads them.

**First run: about 5 minutes** (about 1.8 GB of downloads, with progress bars).
The *"unauthenticated requests to the HF Hub"* warning is harmless. To remove it and get faster downloads, create a free token at huggingface.co → Settings → Access Tokens, then add it in Colab's **Secrets** panel as `HF_TOKEN` with notebook access turned on.

In [ ]:
import io, shutil
import pyarrow.parquet as pq
from PIL import Image as PILImage
from huggingface_hub import HfApi, hf_hub_download
from datasets import Dataset, Features, Value, Image, List, load_from_disk

# name: (repo, split, number of data files to sample from, number of questions).
# Each file holds ~1,700 (TextVQA) or ~3,200 (VQAv2) questions; more files = more variety.
SOURCES = {
    "textvqa_train": ("lmms-lab/textvqa", "train",      3, N_TRAIN),
    "textvqa_eval":  ("lmms-lab/textvqa", "validation", 1, N_EVAL),
    "vqav2_train":   ("lmms-lab/VQAv2",   "train",      3, N_TRAIN),
    "vqav2_eval":    ("lmms-lab/VQAv2",   "validation", 2, N_EVAL),
}
FEATURES = Features({"qid": Value("string"), "image_id": Value("string"), "task": Value("string"),
                     "question": Value("string"), "answers": List(Value("string")), "image": Image()})

def answers_as_strings(answers):
    # TextVQA stores answers as strings; VQAv2 as dicts {"answer": ..., "answer_confidence": ..., "answer_id": ...}
    return [a["answer"] if isinstance(a, dict) else a for a in answers]

def shrink_to_jpeg(image_struct):
    img = PILImage.open(io.BytesIO(image_struct["bytes"])).convert("RGB")
    img.thumbnail((IMG_MAX, IMG_MAX))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return {"bytes": buf.getvalue(), "path": None}

def build_subset(name):
    repo, split, n_files, n = SOURCES[name]
    task = name.split("_")[0]
    rng = random.Random(f"{SEED}-{name}")
    files = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset")
                   if f.startswith(f"data/{split}-") and f.endswith(".parquet"))
    files = sorted(rng.sample(files, n_files))
    print(f"{name}: downloading {n_files} {split} file(s): {[f.split('/')[-1] for f in files]}")
    paths = [hf_hub_download(repo, f, repo_type="dataset") for f in files]

    # 1) list every question in the downloaded files and choose n of them at random
    candidates = []                                   # (file index, row group, row within group)
    for fi, p in enumerate(paths):
        md = pq.ParquetFile(p).metadata
        for rg in range(md.num_row_groups):
            candidates += [(fi, rg, r) for r in range(md.row_group(rg).num_rows)]
    chosen = sorted(rng.sample(candidates, n))
    print(f"   chose {n} of {len(candidates)} questions")
    by_group = {}
    for fi, rg, r in chosen:
        by_group.setdefault((fi, rg), []).append(r)

    # 2) read one row group (~100 questions) at a time, keep only the chosen rows, shrink their images
    def gen():
        t0, done = time.time(), 0
        for (fi, rg), rows in by_group.items():
            tbl = pq.ParquetFile(paths[fi]).read_row_group(rg, columns=["question_id", "image_id", "question", "answers", "image"])
            for r in rows:
                ex = tbl.slice(r, 1).to_pylist()[0]
                yield {"qid": str(ex["question_id"]), "image_id": str(ex["image_id"]), "task": task,
                       "question": ex["question"], "answers": answers_as_strings(ex["answers"]),
                       "image": shrink_to_jpeg(ex["image"])}
                done += 1
                if done % 500 == 0:
                    print(f"   {done}/{n} images processed ({time.time()-t0:.0f}s)", flush=True)
            del tbl
    ds = Dataset.from_generator(gen, features=FEATURES)
    return ds.shuffle(seed=SEED).flatten_indices()     # mix the order so batches aren't grouped by file

def build_or_load(name):
    path, tmp = f"{DATA_DIR}/{name}", f"{DATA_DIR}/{name}__tmp"
    if os.path.exists(path):
        print(f"{name}: already on Drive, loading")
        return load_from_disk(path)
    t = time.time()
    ds = build_subset(name)
    shutil.rmtree(tmp, ignore_errors=True)
    ds.save_to_disk(tmp)                               # write to a temp folder, then rename:
    os.rename(tmp, path)                               # an interrupted save never leaves a half-written subset
    print(f"built {name}: {len(ds)} questions in {time.time()-t:.0f}s")
    return load_from_disk(path)

tvqa_train = build_or_load("textvqa_train")
tvqa_eval  = build_or_load("textvqa_eval")
vqa_train  = build_or_load("vqav2_train")
vqa_eval   = build_or_load("vqav2_eval")

for name, d in [("textvqa_train", tvqa_train), ("textvqa_eval", tvqa_eval), ("vqav2_train", vqa_train), ("vqav2_eval", vqa_eval)]:
    print(f"{name:14s} {len(d):5d} questions, {len(set(d['image_id'])):5d} images")
assert not set(tvqa_train["image_id"]) & set(tvqa_eval["image_id"]), "TextVQA train/eval share images!"
assert not set(vqa_train["image_id"]) & set(vqa_eval["image_id"]), "VQAv2 train/eval share images!"
print("example:", tvqa_eval[0]["question"], "->", tvqa_eval[0]["answers"][:3])

textvqa_train: downloading 3 train file(s): ['train-00006-of-00020.parquet', 'train-00009-of-00020.parquet', 'train-00018-of-00020.parquet']


data/train-00006-of-00020.parquet: reconstructing file:   0%|          |  0.00B /  304MB            

data/train-00006-of-00020.parquet: downloading bytes:           |  0.00B            

data/train-00009-of-00020.parquet: reconstructing file:   0%|          |  0.00B /  299MB            

data/train-00009-of-00020.parquet: downloading bytes:           |  0.00B            

data/train-00018-of-00020.parquet: reconstructing file:   0%|          |  0.00B /  276MB            

... (33 outputs left out)

   500/500 images processed (6s)


Flattening the indices:   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

built vqav2_eval: 500 questions in 14s
textvqa_train   2000 questions,  1719 images
textvqa_eval     500 questions,   439 images
vqav2_train     2000 questions,  1144 images
vqav2_eval       500 questions,   393 images
example: what team name is on the jersey? -> ['lakers', 'lakers', 'lakers']


## 2. Load SmolVLM-256M-Instruct
* `do_image_splitting=False`: by default SmolVLM cuts large images into many tiles (up to about 1,000 image tokens). With tiling off, each image is one 512×512 view, i.e. **64 image tokens**. That makes training several times faster, which we need because we'll run a whole grid of learning rates.
* **fp16** weights, because the T4 has no bf16 support.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID, do_image_splitting=False)
processor.tokenizer.padding_side = "left"      # left-pad for batched generation
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()

n_params = sum(p.numel() for p in model.parameters())
n_vis = sum(p.numel() for n, p in model.named_parameters() if "vision_model" in n)
n_txt = sum(p.numel() for n, p in model.named_parameters() if "text_model" in n or "lm_head" in n)
print(f"total {n_params/1e6:.1f}M params | vision encoder {n_vis/1e6:.1f}M | language model {n_txt/1e6:.1f}M")

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.35k [00:00<?, ?B/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/28.2k [00:00<?, ?B/s]

... (6 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

total 256.5M params | vision encoder 86.4M | language model 163.0M


## 3. Prompting and the VQA accuracy metric
**Prompt.** We append *"Answer the question using a single word or phrase."*, the standard short-answer VQA instruction, so the model replies "stop" instead of "The sign says stop."

**VQA accuracy** (the official metric for both TextVQA and VQAv2). Each question has 10 human answers. A prediction is scored against each subset of 9 of those answers as `min(#matches / 3, 1)`, and the 10 scores are averaged. In practice: if 3 or more people gave your answer, it scores 1.0. Before comparing, answers are normalised: lower-cased, punctuation removed, number words turned into digits, and "a"/"an"/"the" dropped. This follows the official evaluation script.

In [ ]:
PROMPT_SUFFIX = " Answer the question using a single word or phrase."

def build_prompt(question, answer=None):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question + PROMPT_SUFFIX}]}]
    if answer is None:
        return processor.apply_chat_template(msgs, add_generation_prompt=True)
    msgs.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return processor.apply_chat_template(msgs)

# ---- official-style VQA answer normalisation ----
_CONTRACTIONS = {"dont": "don't", "doesnt": "doesn't", "isnt": "isn't", "arent": "aren't", "cant": "can't",
                 "wont": "won't", "didnt": "didn't", "wasnt": "wasn't", "werent": "weren't", "couldnt": "couldn't",
                 "shouldnt": "shouldn't", "wouldnt": "wouldn't", "hasnt": "hasn't", "havent": "haven't",
                 "im": "i'm", "ive": "i've", "youre": "you're", "theyre": "they're", "thats": "that's",
                 "whats": "what's", "wheres": "where's", "hes": "he's", "shes": "she's", "lets": "let's"}
_NUMS = {"none": "0", "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4", "five": "5",
         "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10"}
_ARTICLES = {"a", "an", "the"}
_PUNCT = [";", "/", "[", "]", '"', "{", "}", "(", ")", "=", "+", "\\", "_", "-", ">", "<", "@", "`", ",", "?", "!"]
_PERIOD = re.compile(r"(?!<=\d)(\.)(?!\d)")
_COMMA = re.compile(r"(\d)(,)(\d)")

def normalize_answer(s):
    s = s.replace("\n", " ").replace("\t", " ").strip().lower()
    out = s
    for p in _PUNCT:
        if (p + " " in s or " " + p in s) or _COMMA.search(s):
            out = out.replace(p, "")
        else:
            out = out.replace(p, " ")
    out = _PERIOD.sub("", out)
    words = []
    for w in out.split():
        w = _NUMS.get(w, w)
        if w in _ARTICLES:
            continue
        words.append(_CONTRACTIONS.get(w, w))
    return " ".join(words)

def vqa_score(pred, answers):
    p = normalize_answer(pred)
    gts = [normalize_answer(a) for a in answers]
    scores = []
    for i in range(len(gts)):
        others = gts[:i] + gts[i + 1:]
        scores.append(min(1.0, sum(g == p for g in others) / 3.0))
    return float(np.mean(scores))

# quick sanity checks
assert vqa_score("Stop", ["stop"] * 10) == 1.0
assert vqa_score("two", ["2"] * 10) == 1.0
assert vqa_score("cat", ["dog"] * 10) == 0.0
print(vqa_score("yes", ["yes"] * 2 + ["no"] * 8))   # 2 of 10 humans agree -> 0.6

0.6


## 4. Batched generation and evaluation
Greedy decoding (`do_sample=False`) keeps evaluation deterministic. We save every prediction to Drive. You'll need them later for the report and for looking at examples.

In [ ]:
@torch.no_grad()
def predict(model, data, batch_size=32, max_new_tokens=10):
    model.eval()
    preds = []
    for i in range(0, len(data), batch_size):
        chunk = data[i:i + batch_size]                     # dict of lists
        texts = [build_prompt(q) for q in chunk["question"]]
        images = [[im] for im in chunk["image"]]
        batch = processor(text=texts, images=images, return_tensors="pt", padding=True).to(model.device)
        batch["pixel_values"] = batch["pixel_values"].to(model.dtype)
        out = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False)
        new = out[:, batch["input_ids"].shape[1]:]
        preds += [s.strip() for s in processor.batch_decode(new, skip_special_tokens=True)]
    return preds

def evaluate(model, data, tag, batch_size=32):
    t = time.time()
    preds = predict(model, data, batch_size=batch_size)
    scores = [vqa_score(p, a) for p, a in zip(preds, data["answers"])]
    acc = 100 * float(np.mean(scores))
    se = 100 * float(np.std(scores, ddof=1) / np.sqrt(len(scores)))   # standard error -> report acc ± 1.96*se
    with open(f"{RES_DIR}/preds_{tag}.jsonl", "w") as f:
        for q, p, a, s in zip(data["question"], preds, data["answers"], scores):
            f.write(json.dumps({"question": q, "pred": p, "answers": a, "score": s}) + "\n")
    print(f"{tag:28s} VQA acc = {acc:5.1f} ± {1.96*se:.1f} (95% CI)   [{len(scores)} q, {time.time()-t:.0f}s]")
    return acc, se, preds

## 5. Zero-shot baseline
This is the first number for your results table. It also tests the assumption behind **H2**: if TextVQA is "harder" for SmolVLM-256M, its zero-shot accuracy should be clearly **lower** than VQAv2's.

In [ ]:
results = {}
for tag, data in [("zeroshot_textvqa", tvqa_eval), ("zeroshot_vqav2", vqa_eval)]:
    acc, se, preds = evaluate(model, data, tag)
    results[tag] = {"acc": acc, "se": se, "n": len(data)}
    for q, p, a in list(zip(data["question"], preds, data["answers"]))[:5]:
        print(f"   Q: {q}\n   pred: {p!r}   humans: {a[:4]}")

with open(f"{RES_DIR}/zeroshot.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

zeroshot_textvqa             VQA acc =  32.8 ± 4.0 (95% CI)   [500 q, 84s]
   Q: what team name is on the jersey?
   pred: 'Lakers.'   humans: ['lakers', 'lakers', 'lakers', 'lakers']
   Q: what brand is advertised on the wall?
   pred: 'LG.'   humans: ['lg whisen', 'lg', '11', 'lg']
   Q: what number is on the red jersey?
   pred: '8.'   humans: ['6', '6', '6', '6']
   Q: what is the flavor of the product?
   pred: 'Sour apple.'   humans: ['sour apple', 'sour apple', 'sour apple', 'sour apple']
   Q: what is steel-enforced on this garage door opener?
   pred: 'Chambler.'   humans: ['belt', 'belt', 'belt', 'belt']
zeroshot_vqav2               VQA acc =  45.3 ± 4.2 (95% CI)   [500 q, 87s]
   Q: How many mountains are in the scene?
   pred: '2.'   humans: ['0', '3', '2 hills', '0']
   Q: What is the reflecting object called?
   pred: 'Mirror.'   humans: ['mirror', 'mirror', 'mirror', 'mirror']
   Q: What is the man doing?
   pred: 'Skateboarding.'   humans: ['skateboarding', 'skateboardi